# 01 - Audit of the raw data

Goal: understand each raw file before any analysis. Write what you find under each step.

In [43]:
import sys
sys.path.append("../scripts")   # so we can import load_data.py

import pandas as pd
from load_data import load_all, uhf42

data = load_all()
list(data)

['pm25', 'asthma_ed_5to17', 'asthma_ed_adults', 'poverty']

## 1. pm25

In [44]:
data["pm25"].head()

,TimePeriod,GeoTypePretty,GeoTypeDesc,GeoID,GeoRank,BoroID,Borough,Geography,Area,Annual mean mcg/m3,Summer mean mcg/m3,Winter mean mcg/m3,flagged
0,2025,Citywide,Citywide,1,0,-,-,New York City,New York City,6.3,8.0,6.4,False
1,2025,Borough,Borough,1,1,1,Bronx,Bronx,Bronx,6.5,8.1,6.5,False
2,2025,Borough,Borough,2,1,2,Brooklyn,Brooklyn,Brooklyn,6.3,8.1,6.4,False
3,2025,Borough,Borough,3,1,3,Manhattan,Manhattan,Manhattan,7.4,9.0,7.5,False
4,2025,Borough,Borough,4,1,4,Queens,Queens,Queens,6.3,8.0,6.3,False


**What I notice:**

Each row is one area in one year, and areas include citywide, boroughs, community districts and UHF neighborhoods all mixed in one file. The measures are annual, summer and winter mean PM2.5. The first rows are 2025 boroughs, so the file starts with the newest year. Before using it I need to filter to UHF 42 and the years 2015-2023.



## 2. asthma_ed_5to17

In [45]:
data["asthma_ed_5to17"].head()

,TimePeriod,GeoTypePretty,GeoTypeDesc,GeoID,GeoRank,BoroID,Borough,Geography,Area,Number,"Estimated annual rate per 10,000",flagged
0,2023,Citywide,Citywide,1,0,-,-,New York City,New York City,16563,139.1,False
1,2023,Borough,Borough,1,1,1,Bronx,Bronx,Bronx,5638,234.3,False
2,2023,Borough,Borough,2,1,2,Brooklyn,Brooklyn,Brooklyn,4930,123.7,False
3,2023,Borough,Borough,3,1,3,Manhattan,Manhattan,Manhattan,2552,163.7,False
4,2023,Borough,Borough,4,1,4,Queens,Queens,Queens,3003,94.9,False


**What I notice:**





## 3. asthma_ed_adults

In [46]:
data["asthma_ed_adults"].head()

,TimePeriod,GeoTypePretty,GeoTypeDesc,GeoID,GeoRank,BoroID,Borough,Geography,Area,Number,"Estimated annual rate per 10,000","Age-adjusted rate per 10,000",flagged
0,2023,Citywide,Citywide,1,0,-,-,New York City,New York City,41772.0,61.9,63.5,False
1,2023,Borough,Borough,1,1,1,Bronx,Bronx,Bronx,13834.0,132.2,133.6,False
2,2023,Borough,Borough,2,1,2,Brooklyn,Brooklyn,Brooklyn,12529.0,61.6,63.3,False
3,2023,Borough,Borough,3,1,3,Manhattan,Manhattan,Manhattan,7258.0,51.3,52.6,False
4,2023,Borough,Borough,4,1,4,Queens,Queens,Queens,6577.0,35.4,36.9,False


**What I notice:**

The adults file has an extra measure that the 5–17 file doesn't: Age-adjusted rate per 10,000



## 4. poverty

In [47]:
data["poverty"].head()

,TimePeriod,GeoTypePretty,GeoTypeDesc,GeoID,GeoRank,BoroID,Borough,Geography,Area,Number,Percent,flagged
0,2019-23,Citywide,Citywide,1,0,-,-,New York City,New York City,1454320,17.4,False
1,2019-23,Borough,Borough,1,1,1,Bronx,Bronx,Bronx,374138,27.0,False
2,2019-23,Borough,Borough,2,1,2,Brooklyn,Brooklyn,Brooklyn,494754,18.9,False
3,2019-23,Borough,Borough,3,1,3,Manhattan,Manhattan,Manhattan,250056,15.8,False
4,2019-23,Borough,Borough,4,1,4,Queens,Queens,Queens,282195,12.2,False


**What I notice:**

Asthma and PM2.5 have one row per year, but poverty has one row per 4-year window.



In [48]:
for name, df in data.items():
    u = uhf42(df)
    print(f"{name}: {len(u)} rows, {u['GeoID'].nunique()} areas")

pm25: 462 rows, 42 areas
asthma_ed_5to17: 798 rows, 42 areas
asthma_ed_adults: 756 rows, 42 areas
poverty: 546 rows, 42 areas


In [49]:
for name, df in data.items():
    years = sorted(uhf42(df)["TimePeriod"].unique())
    print(name, years[0], "->", years[-1], f"({len(years)} periods)")

expected = [str(y) for y in range(2015, 2024)]
for name in ["pm25", "asthma_ed_5to17", "asthma_ed_adults"]:
    have = set(uhf42(data[name])["TimePeriod"])
    print(name, "missing in 2015-2023:", [y for y in expected if y not in have])

pm25 2015 -> 2025 (11 periods)
asthma_ed_5to17 2005 -> 2023 (19 periods)
asthma_ed_adults 2005 -> 2023 (18 periods)
poverty 2007-11 -> 2019-23 (13 periods)
pm25 missing in 2015-2023: []
asthma_ed_5to17 missing in 2015-2023: []
asthma_ed_adults missing in 2015-2023: ['2015']


In [50]:
for name, df in data.items():
    u = uhf42(df)
    nulls = u.isna().sum()
    print(name, "NaN:", nulls[nulls > 0].to_dict())
    if u["flagged"].any():
        print(u.loc[u["flagged"], ["TimePeriod", "Geography"]].to_string(index=False))

pm25 NaN: {}
asthma_ed_5to17 NaN: {}
TimePeriod                Geography
      2021 Greenwich Village - SoHo
      2020 Greenwich Village - SoHo
      2020    Bayside - Little Neck
asthma_ed_adults NaN: {}
poverty NaN: {}


In [51]:
for name, df in data.items():
    u = uhf42(df)
    dup = u.duplicated(subset=["GeoID", "TimePeriod"]).sum()
    per_area = u.groupby("GeoID").size()
    print(name, "duplicate rows:", dup, "| rows per area min/max:", per_area.min(), per_area.max())

pm25 duplicate rows: 0 | rows per area min/max: 11 11
asthma_ed_5to17 duplicate rows: 0 | rows per area min/max: 19 19
asthma_ed_adults duplicate rows: 0 | rows per area min/max: 18 18
poverty duplicate rows: 0 | rows per area min/max: 13 13


In [52]:
names = {n: uhf42(df).drop_duplicates("GeoID").set_index("GeoID")["Geography"]
         for n, df in data.items()}
ref = names["pm25"]
for n, s in names.items():
    print(n, "same IDs:", set(s.index) == set(ref.index),
          "| name mismatches:", int((s.reindex(ref.index) != ref).sum()))

pm25 same IDs: True | name mismatches: 0
asthma_ed_5to17 same IDs: True | name mismatches: 0
asthma_ed_adults same IDs: True | name mismatches: 0
poverty same IDs: True | name mismatches: 0


In [53]:
for name, df in data.items():
    print(name)
    display(uhf42(df).describe().T)

pm25


,count,mean,std,min,25%,50%,75%,max
GeoID,462.0,288.428571,123.230602,101.0,204.0,303.5,404.0,504.0
GeoRank,462.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0
Annual mean mcg/m3,462.0,7.135931,1.181311,5.0,6.3,6.8,7.8,11.7
Summer mean mcg/m3,462.0,8.442208,1.172633,5.5,7.7,8.3,9.2,12.5
Winter mean mcg/m3,462.0,7.645238,1.373837,5.0,6.6,7.4,8.4,12.6


asthma_ed_5to17


,count,mean,std,min,25%,50%,75%,max
GeoID,798.0,288.428571,123.174364,101.0,204.0,303.5,404.0,504.0
GeoRank,798.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0
Number,798.0,528.151629,519.526054,4.0,133.25,313.0,846.75,2147.0
"Estimated annual rate per 10,000",798.0,166.009774,131.482633,3.7,71.525,132.15,224.4,687.1


asthma_ed_adults


,count,mean,std,min,25%,50%,75%,max
GeoID,756.0,288.428571,123.178657,101.0,204.0,303.5,404.0,504.0
GeoRank,756.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0
Number,756.0,1383.736772,1220.664325,32.0,462.0,945.5,2009.25,6088.0
"Estimated annual rate per 10,000",756.0,91.757275,81.764591,4.2,34.075,59.85,119.675,375.4
"Age-adjusted rate per 10,000",756.0,93.115608,80.713457,4.3,35.775,63.85,120.175,376.3


poverty


,count,mean,std,min,25%,50%,75%,max
GeoID,546.0,288.428571,123.210045,101.0,204.0,303.5,404.0,504.0
GeoRank,546.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0
Number,546.0,37425.177656,23627.234961,5609.0,17624.75,32998.0,54513.75,92821.0
Percent,546.0,18.564103,9.066128,5.5,11.725,16.05,24.15,43.1
